In [1]:
!pip -q install datasets librosa scikit-learn

In [2]:
import numpy as np
import librosa
import tensorflow as tf
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [3]:
ds = load_dataset("SpeechAntiSpoofingBenchmarks/DFADD", split="test")

real = [i for i, row in enumerate(ds) if row["label"] == 0]
fake = [i for i, row in enumerate(ds) if row["label"] == 1]

rng = np.random.default_rng(42)
real = rng.choice(real, 755, replace=False)
fake = rng.choice(fake, 755, replace=False)

indices = np.concatenate([real, fake])
rng.shuffle(indices)

ds = ds.select(indices.tolist())

print("Total clips:", len(ds))
print("Real:", sum(row["label"] == 0 for row in ds))
print("Spoof:", sum(row["label"] == 1 for row in ds))

README.md:   0%|          | 0.00/3.70k [00:00<?, ?B/s]

data/test-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  302MB            

data/test-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/test-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  239MB            

data/test-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/3755 [00:00<?, ? examples/s]

Total clips: 1510
Real: 755
Spoof: 755


In [4]:
IMG_SIZE = 128

X, y = [], []

for row in ds:
    audio = row["audio"]
    signal = audio["array"]
    sr = audio["sampling_rate"]

    mel = librosa.feature.melspectrogram(
        y=signal.astype(np.float32),
        sr=sr,
        n_mels=128
    )

    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-8)
    mel = tf.image.resize(mel[..., None], (IMG_SIZE, IMG_SIZE)).numpy()

    X.append(np.repeat(mel, 3, axis=-1) * 255)
    y.append(row["label"])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print(X.shape, y.shape)

(1510, 128, 128, 3) (1510,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (1208, 128, 128, 3)
Testing: (302, 128, 128, 3)


In [11]:
MODEL = "AlexNet"

In [12]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16, VGG19, InceptionV3, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_pre
from tensorflow.keras.applications.vgg19 import preprocess_input as vgg19_pre
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre

def head(x):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    return layers.Dense(1, activation="sigmoid")(x)

def conv(x, filters, size=3, stride=1):
    x = layers.Conv2D(filters, size, strides=stride, padding="same", activation="relu")(x)
    return x

def inception_block(x, filters):
    a, b, c, d = filters
    p1 = conv(x, a, 1)
    p2 = conv(conv(x, b, 1), b, 3)
    p3 = conv(conv(x, c, 1), c, 5)
    p4 = layers.MaxPooling2D(3, strides=1, padding="same")(x)
    p4 = conv(p4, d, 1)
    return layers.Concatenate()([p1, p2, p3, p4])

def build_model(name):
    inputs = layers.Input((128, 128, 3))

    if name in ["VGG16", "VGG19", "Inception", "ResNet"]:
        if name == "VGG16":
            base = VGG16(weights="imagenet", include_top=False, input_shape=(128,128,3))
            x = vgg16_pre(inputs)
        elif name == "VGG19":
            base = VGG19(weights="imagenet", include_top=False, input_shape=(128,128,3))
            x = vgg19_pre(inputs)
        elif name == "Inception":
            base = InceptionV3(weights="imagenet", include_top=False, input_shape=(128,128,3))
            x = inception_pre(inputs)
        else:
            base = ResNet50(weights="imagenet", include_top=False, input_shape=(128,128,3))
            x = resnet_pre(inputs)

        base.trainable = False
        outputs = head(base(x, training=False))

    else:
        x = layers.Rescaling(1./255)(inputs)

        if name == "LeNet":
            x = conv(x, 6, 5)
            x = layers.AveragePooling2D(2)(x)
            x = conv(x, 16, 5)
            x = layers.AveragePooling2D(2)(x)
            x = layers.Flatten()(x)
            x = layers.Dense(120, activation="relu")(x)
            x = layers.Dense(84, activation="relu")(x)
            outputs = layers.Dense(1, activation="sigmoid")(x)

        elif name == "AlexNet":
            x = conv(x, 32, 11, 4)
            x = layers.MaxPooling2D(3, 2)(x)
            x = conv(x, 64, 5)
            x = layers.MaxPooling2D(3, 2)(x)
            x = conv(x, 128)
            x = conv(x, 128)
            x = layers.GlobalAveragePooling2D()(x)
            x = layers.Dense(128, activation="relu")(x)
            x = layers.Dropout(0.4)(x)
            outputs = layers.Dense(1, activation="sigmoid")(x)

        elif name == "GoogLeNet":
            x = conv(x, 32, 7, 2)
            x = layers.MaxPooling2D(3, 2, padding="same")(x)
            x = inception_block(x, [16, 16, 8, 8])
            x = inception_block(x, [24, 24, 12, 12])
            x = layers.MaxPooling2D(2)(x)
            x = inception_block(x, [32, 24, 12, 12])
            outputs = head(x)

        else:
            raise ValueError("Unknown model name")

    model = Model(inputs, outputs)
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [13]:
model = build_model(MODEL)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 14s 375ms/step - accuracy: 0.5735 - loss: 0.6695 - val_accuracy: 0.7769 - val_loss: 0.5266
Epoch 2/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 9s 280ms/step - accuracy: 0.7422 - loss: 0.5604 - val_accuracy: 0.8140 - val_loss: 0.4201
Epoch 3/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 9s 234ms/step - accuracy: 0.7795 - loss: 0.4481 - val_accuracy: 0.8140 - val_loss: 0.4138
Epoch 4/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 10s 232ms/step - accuracy: 0.8375 - loss: 0.4040 - val_accuracy: 0.8264 - val_loss: 0.3530
Epoch 5/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 12s 273ms/step - accuracy: 0.8458 - loss: 0.3515 - val_accuracy: 0.8595 - val_loss: 0.3212
Epoch 6/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 9s 286ms/step - accuracy: 0.8696 - loss: 0.2990 - val_accuracy: 0.8388 - val_loss: 0.3177
Epoch 7/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 7s 236ms/step - accuracy: 0.8571 - loss: 0.3321 - val_accuracy: 0.8843 - val_loss: 0.2742
Epoch 8/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 11s 247ms/step - accuracy: 0.9099 - loss: 0.2243 - val_accuracy

In [14]:
prob = model.predict(X_test).ravel()
pred = (prob >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred, zero_division=0))
print("Recall:", recall_score(y_test, pred, zero_division=0))
print("F1-score:", f1_score(y_test, pred, zero_division=0))
print(classification_report(y_test, pred, target_names=["Real", "Spoof"], zero_division=0))

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step
Accuracy: 0.9337748344370861
Precision: 0.901840490797546
Recall: 0.9735099337748344
F1-score: 0.9363057324840764
              precision    recall  f1-score   support

        Real       0.97      0.89      0.93       151
       Spoof       0.90      0.97      0.94       151

    accuracy                           0.93       302
   macro avg       0.94      0.93      0.93       302
weighted avg       0.94      0.93      0.93       302



In [15]:
model.save(f"VoiceGuard_{MODEL}.keras")

with open("VoiceGuard_results.txt", "a") as f:
    f.write(
        f"{MODEL}: Accuracy={accuracy_score(y_test, pred):.4f}, "
        f"Precision={precision_score(y_test, pred, zero_division=0):.4f}, "
        f"Recall={recall_score(y_test, pred, zero_division=0):.4f}, "
        f"F1={f1_score(y_test, pred, zero_division=0):.4f}\n"
    )

print("Saved:", f"VoiceGuard_{MODEL}.keras")

Saved: VoiceGuard_AlexNet.keras
